# Create BrightFocus Foundation Awards

Creates BrightFocus Foundation awards from the foundation's own public
**Grant Search** (https://www.brightfocus.org/explore-research/). The search is
backed by the WordPress `grant` post type; the script enumerates every grant
through the public WP REST API (`/wp-json/wp/v2/grant`, X-WP-Total = 1,535 on
2026-10-01) and reads each grant page's details block: Program, Award Type,
Award Amount (USD), Active Dates, **Grant ID**, Acknowledgement, Mentor(s),
Goals / Summary, plus the Principal Investigator card (name, institution,
city/state/country). Three programs: Alzheimer's Disease Research, Macular
Degeneration Research, National Glaucoma Research (1994-2026).

**1,534 grants parsed, 1,533 shipped, 1994-2026, USD 288.2M, 100% title /
amount / dates, 99.8% PI** (local run 2026-10-01). One post
(`abnormal-aspartyl-residues-and-amyloid-formation-2`) is unrenderable on the
site (301 redirect loop between its two slugs) and is missing.

**Scope (batch rule: exclude only money that is clearly not research):**
excluded `M2009900` "Macular Degeneration Support Group" (USD 22,000, patient
support group), kept in the parquet with `excluded = '1'`. Kept but flagged
(research-ecosystem support, not projects): journal / society / meeting /
prize lines `Z2010999`, `CA2021010` (Molecular Neurodegeneration journal,
USD 5.65M), `CA2017563`, `CA2021011`, `CG2017222`, `CA2017215`. Three `C...`
contract grants name an organisation as PI (Sage Bionetworks, RexCeptor, ISER);
the script leaves their person name NULL, so `lead_investigator` is NULL.

**Prerequisites:**
- Run `scripts/local/brightfocus_to_s3.py` first. It uploads
  `s3://openalex-ingest/awards/brightfocus/brightfocus_projects.parquet` (§1.4 shrink guard).

**`funder_award_id` (§2.1.1):** the page's **Grant ID** (e.g. `A2021036S`,
`G2016023`, `M2026014N`; first letter = program A/G/M, then award year). This is
the reference BrightFocus grantees cite: the crossref_work.grants /
crossref_work_funders / europepmc_work_funders shells for F4320308990 carry
`A2021036S`, `G2016023`, `M2012044`, ... BrightFocus registers no grant DOIs
with Crossref (0 `type:grant` deposits for 10.13039/100006312 or
10.13039/100000967 on 2026-10-01).

**Twin:** American Health Assistance Foundation (F4320306229, same ROR
03cvfxv40) is BrightFocus's pre-2013 name. Its citation shells use the same
Grant ID form (`A2007-05`, `G2006-118`). This ingest ships under the tracker's
F4320308990 only; the overlap cell below reports how many AHAF shells the same
Grant IDs would collapse onto.

**Currency:** amounts are printed in USD on every page (US charity).

**Funder details (Path A, F4320* Crossref-registered):**
- funder_id: `4320308990`
- display_name / ror_id / doi: from `openalex.funders.funders` (ror 03cvfxv40, doi 10.13039/100006312)

**Priority:** `562` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.brightfocus_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/brightfocus/brightfocus_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total_grants, COUNT(DISTINCT funder_award_id) as distinct_ids,
       COUNT(excluded) as excluded_out_of_scope
FROM openalex.awards.brightfocus_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.brightfocus_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.brightfocus_raw LIMIT 5;

## Step 1.6: Funder existence check (Path A)
F4320308990 is a Crossref-registered funder, so it MUST resolve to exactly 1
row in `openalex.funders.funders`. If 0 rows, STOP (do not proceed) and flag.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.funders.funders
WHERE funder_id = 4320308990;

## Step 2: Create BrightFocus Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.brightfocus_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320308990  -- BrightFocus Foundation
),

parsed AS (
    SELECT
        g.*,
        -- struct field order MUST match openalex_awards_raw: ...orcid, role_start, affiliation.
        CASE WHEN NULLIF(TRIM(g.lead_family_name), '') IS NOT NULL THEN named_struct(
            'given_name', NULLIF(TRIM(g.lead_given_name), ''),
            'family_name', NULLIF(TRIM(g.lead_family_name), ''),
            'orcid', CAST(NULL AS STRING),
            'role_start', CAST(NULL AS DATE),
            'affiliation', named_struct(
                'name', NULLIF(TRIM(g.lead_institution), ''),
                'country', NULLIF(TRIM(g.lead_country), ''),
                'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
            )
        ) END AS lead
    FROM openalex.awards.brightfocus_raw g
)

SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000 as id,
    NULLIF(TRIM(g.title), '') as display_name,
    NULLIF(TRIM(g.description), '') as description,
    f.funder_id,
    TRIM(g.funder_award_id) as funder_award_id,
    TRY_CAST(g.amount AS DOUBLE) as amount,
    CASE WHEN TRY_CAST(g.amount AS DOUBLE) IS NOT NULL THEN 'USD' END as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    CASE WHEN LOWER(g.award_type) LIKE '%fellow%' THEN 'fellowship' ELSE 'research' END as funding_type,
    -- program + award type, e.g. "Alzheimer's Disease Research: Postdoctoral Fellowship"
    CONCAT_WS(': ', NULLIF(TRIM(g.program), ''), NULLIF(TRIM(g.award_type), '')) as funder_scheme,
    'brightfocus_grant_search' as provenance,
    TRY_TO_DATE(g.start_date, 'yyyy-MM-dd') as start_date,
    TRY_TO_DATE(g.end_date, 'yyyy-MM-dd') as end_date,
    YEAR(TRY_TO_DATE(g.start_date, 'yyyy-MM-dd')) as start_year,
    YEAR(TRY_TO_DATE(g.end_date, 'yyyy-MM-dd')) as end_year,
    g.lead as lead_investigator,
    CAST(NULL AS STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>) as co_lead_investigator,
    CASE WHEN g.lead IS NOT NULL THEN array(g.lead) END as investigators,
    g.landing_page_url as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM parsed g
CROSS JOIN src_funder f
WHERE g.funder_award_id IS NOT NULL AND TRIM(g.funder_award_id) != ''
  AND g.excluded IS NULL;  -- scope exclusions (see header)

In [ ]:
%sql
-- Shape check (§2.1.1): every shipped id is a BrightFocus Grant ID
-- (program letter(s) A/G/M/Z or contract C/CA/CG/CM + 4-digit year + 3-4 digits + optional suffix letter); expect 0 rows.
SELECT funder_award_id, display_name, landing_page_url
FROM openalex.awards.brightfocus_awards
WHERE NOT funder_award_id RLIKE '^C?[AGMZ]?[0-9]{7,8}[A-Z]?$';

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'brightfocus_grant_search' AND priority = 562;

-- Insert into openalex_awards_raw with priority.
-- Priority 562: direct-from-funder ingest of BrightFocus's own Grant Search.
-- Higher wins under the 2026-06-20 DESC dedup (oxjob #500), so 562 outranks
-- the acknowledgement shells (priority 0) that share these Grant IDs.
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    562 as priority
FROM openalex.awards.brightfocus_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_awards, COUNT(DISTINCT id) as distinct_ids
FROM openalex.awards.brightfocus_awards;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, amount, currency, start_date, end_date,
       lead_investigator.given_name, lead_investigator.family_name,
       lead_investigator.affiliation.name, lead_investigator.affiliation.country
FROM openalex.awards.brightfocus_awards LIMIT 20;

In [ ]:
%sql
SELECT funder_scheme, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_usd
FROM openalex.awards.brightfocus_awards
GROUP BY funder_scheme ORDER BY cnt DESC;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_usd
FROM openalex.awards.brightfocus_awards
WHERE start_year IS NOT NULL GROUP BY start_year ORDER BY start_year;

In [ ]:
%sql
-- Section 6.4a frequency check: PI top-20 must be a real long-tail.
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.brightfocus_awards
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
SELECT display_name, COUNT(*) AS n
FROM openalex.awards.brightfocus_awards
GROUP BY 1 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- Section 6.3 / 6.7 coverage.
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_description,
    COUNT(amount) as has_amount,
    COUNT(DISTINCT currency) as distinct_currencies,
    ROUND(MIN(amount), 0) as min_amt,
    ROUND(MAX(amount), 0) as max_amt,
    ROUND(AVG(amount), 0) as avg_amt,
    COUNT(start_date) as has_start_date,
    COUNT(end_date) as has_end_date,
    COUNT(lead_investigator.family_name) as has_pi_name,
    COUNT(lead_investigator.affiliation.name) as has_institution
FROM openalex.awards.brightfocus_awards;

In [ ]:
%sql
-- Overlap with existing acknowledgement shells: same award id = these upgrade in CreateAwards dedup.
-- Second half: AHAF twin (F4320306229) shells that the same Grant ID would hit if keyed to the twin.
SELECT 'F4320308990 shells collapsed' as what, c.provenance, COUNT(*) as shells, COUNT(t.id) as matched
FROM openalex.awards.openalex_awards_raw c
LEFT JOIN openalex.awards.brightfocus_awards t ON t.id = c.id
WHERE c.funder_id = 4320308990 AND c.priority = 0
GROUP BY c.provenance
UNION ALL
SELECT 'AHAF twin F4320306229 (same Grant ID)' as what, c.provenance, COUNT(*) as shells, COUNT(t.funder_award_id) as matched
FROM openalex.awards.openalex_awards_raw c
LEFT JOIN openalex.awards.brightfocus_awards t
  ON abs(xxhash64(CONCAT(4320306229, ':', LOWER(TRIM(t.funder_award_id))))) % 9000000000 = c.id
WHERE c.funder_id = 4320306229 AND c.priority = 0
GROUP BY c.provenance;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'brightfocus_grant_search'
GROUP BY provenance, priority;